In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Activation, Dropout, Flatten, Dense
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.utils import Sequence

import tensorflow as tf
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from PIL import Image
import os

# Сплитим фотки для обучения и валидации модели

In [25]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

# ищем файлы
true_files = sorted((DATA_DIR / "true").glob("*.JPG"))
false_files = sorted((DATA_DIR / "false").glob("*.JPG"))

# создаем массив из индексов данных
X_full = np.array(true_files + false_files, dtype=object)
# создаем массив из классов 1 - true/ 0 - false
y_full = np.array([1]*len(true_files) + [0]*len(false_files), dtype=int)

# сплитим все пути файлов/метки на 80% (train) и 20% (tmp), чтобы получить выборку для обучения X_train, y_train
X_train, X_tmp, y_train, y_tmp = train_test_split(X_full, y_full, test_size=0.2, random_state=42, stratify=y_full)

# сплитим 20% (tmp) на пути файлов/меток на 10% (val) и 10% (test), чтобы получить выборку для валидации X_val и тестов y_val
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)

# проверяем как выполнился сплит
print("Total:", len(X_full), "pos%:", y_full.mean())
print("Train:", len(X_train), "pos%:", y_train.mean())
print("Val:  ", len(X_val), "pos%:", y_val.mean())
print("Test: ", len(X_test), "pos%:", y_test.mean())

Total: 5000 pos%: 0.5
Train: 4000 pos%: 0.5
Val:   500 pos%: 0.5
Test:  500 pos%: 0.5


# Preprocessor

In [26]:
# ресайз входных фото, размер бача, форма тензора
img_width, img_height = 150, 150
batch_size = 10
input_shape = (img_width, img_height, 3)

# создаем препроцессор: генератор изображений из списков путей X и меток y.
class SimpleImageDataset(Sequence):
    def __init__(self, X_paths, y_labels, batch_size=10, shuffle=True):
        self.X = np.array(X_paths)
        self.y = np.array(y_labels)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.X))
        self.on_epoch_end()
    
    def __len__(self):
        # количество батчей за эпоху
        return int(np.ceil(len(self.X) / self.batch_size))
    
    def __getitem__(self, idx):
        # получаем индекс батча
        batch_idx = self.indices[idx*self.batch_size : (idx+1)*self.batch_size]
        batch_X = self.X[batch_idx]
        batch_y = self.y[batch_idx]
        
        # загрузка и преобразование изображений
        images = []
        for path in batch_X:
            img = Image.open(path).convert("RGB")             # читаем JPG, 3 канала
            img = img.resize((img_width, img_height))         # ресайз
            img = np.array(img, dtype=np.float32) / 255.0    # нормализация
            images.append(img)
        return np.array(images), np.array(batch_y, dtype=np.float32)
    
    def on_epoch_end(self):
        # перемешивание после каждой эпохи
        if self.shuffle:
            np.random.shuffle(self.indices)

# создаем три генератора train\val\test
train_generator = SimpleImageDataset(X_train, y_train, batch_size=batch_size, shuffle=True)
val_generator   = SimpleImageDataset(X_val, y_val, batch_size=batch_size, shuffle=False)
test_generator  = SimpleImageDataset(X_test, y_test, batch_size=batch_size, shuffle=False)

# Создаем сверточную нейронную сеть

In [29]:
# параметры нейронной сети
def build_model(input_shape):
    model = Sequential()

    # 1 слой свертки, размер ядра 3х3, количество карт признаков - 32 шт., функция активации ReLU
    # 2 слой подвыборки, выбор максимального значения из квадрата 2х2
    model.add(Conv2D(32, (3, 3), input_shape=input_shape))
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # 3 слой свертки, размер ядра 3х3, количество карт признаков - 32 шт., функция активации ReLU
    # 4 слой подвыборки, выбор максимального значения из квадрата 2х2
    model.add(Conv2D(32, (3, 3)))
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # 5 слой свертки, размер ядра 3х3, количество карт признаков - 64 шт., функция активации ReLU.
    # 6 слой подвыборки, выбор максимального значения из квадрата 2х2
    model.add(Conv2D(64, (3, 3)))
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # 7 слой преобразования из двумерного в одномерное представление
    # 8 полносвязный слой, 64 нейрона, функция активации ReLU
    # 9 слой Dropout (регуляризация, чтобы снизить переобучение)
    # 10 выходной слой, 1 нейрон, функция активации sigmoid
    model.add(Flatten())
    model.add(Dense(64))
    model.add(Activation("relu"))
    model.add(Dropout(0.5))
    model.add(Dense(1))
    model.add(Activation("sigmoid"))

    return model

model = build_model(input_shape)

model.summary()

c:\Users\JohnOnGear\Desktop\Nut_Classify_CNN\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 148, 148, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 72, 72, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 72, 72, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 34, 34, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 34, 34, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │     1,183,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 1)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,212,513 (4.63 MB)

 Trainable params: 1,212,513 (4.63 MB)

 Non-trainable params: 0 (0.00 B)

# Baseline: два обучения, сравнение и выбор epochs 10 и 20

In [30]:
for epochs in [10, 20]:
    # создаем и компилируем модель заново (чтобы не делать повторное обучение на ранее полученных весах)
    model = build_model(input_shape)
    model.compile(
        loss="binary_crossentropy",
        optimizer=Adam(learning_rate=1e-3),
        metrics=["accuracy"]
    )
    
    # обучение на train_generator с валидацией на val_generator
    model.fit(
        train_generator,
        epochs=epochs,
        validation_data=val_generator,
        verbose=1
    )
    
    # оценка на test_generator
    loss, accuracy = model.evaluate(test_generator, verbose=0)
    print(f"Точность на тесте после {epochs} эпох: {accuracy:.4f} / Потери: {loss:.4f}")

c:\Users\JohnOnGear\Desktop\Nut_Classify_CNN\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 179s 446ms/step - accuracy: 0.7360 - loss: 0.4849 - val_accuracy: 0.9160 - val_loss: 0.2893
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 176s 439ms/step - accuracy: 0.9103 - loss: 0.2376 - val_accuracy: 0.9540 - val_loss: 0.1293
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 174s 434ms/step - accuracy: 0.9273 - loss: 0.1981 - val_accuracy: 0.9560 - val_loss: 0.1313
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 175s 438ms/step - accuracy: 0.9588 - loss: 0.1236 - val_accuracy: 0.9740 - val_loss: 0.0889
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 173s 434ms/step - accuracy: 0.9620 - loss: 0.1047 - val_accuracy: 0.9740 - val_loss: 0.0668
Epoch 6/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 190s 475ms/step - accuracy: 0.9755 - loss: 0.0751 - val_accuracy: 0.9740 - val_loss: 0.0755
Epoch 7/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 188s 470ms/step - accuracy: 0.9728 - loss: 0.0778 - val_accuracy: 0.9820 - val_loss: 0.0462
Epoch 8/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 186s 466ms/step - accuracy: 0.9783 -

c:\Users\JohnOnGear\Desktop\Nut_Classify_CNN\venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Точность на тесте после 10 эпох: 0.9980 | Потери: 0.0105
Epoch 1/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 176s 437ms/step - accuracy: 0.6712 - loss: 0.5565 - val_accuracy: 0.9520 - val_loss: 0.1575
Epoch 2/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 172s 431ms/step - accuracy: 0.9340 - loss: 0.1926 - val_accuracy: 0.9800 - val_loss: 0.0561
Epoch 3/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 172s 431ms/step - accuracy: 0.9680 - loss: 0.1043 - val_accuracy: 0.9840 - val_loss: 0.0459
Epoch 4/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 173s 432ms/step - accuracy: 0.9753 - loss: 0.0827 - val_accuracy: 0.9900 - val_loss: 0.0224
Epoch 5/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 175s 439ms/step - accuracy: 0.9507 - loss: 0.1328 - val_accuracy: 0.9820 - val_loss: 0.0409
Epoch 6/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 174s 436ms/step - accuracy: 0.9728 - loss: 0.0679 - val_accuracy: 0.9980 - val_loss: 0.0084
Epoch 7/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 176s 439ms/step - accuracy: 0.9703 - loss: 0.0936 - val_accuracy: 0.9840 - val_loss: 0.0389
Epoch 8/20
400/400 

### Результаты на на тесте после 20 эпох: 1.0000 | Потери: 0.0033 - модель ПЕРЕобучилась. BEST_BATCH = 10 (точность на тесте после 10 эпох: 0.9980 | Потери: 0.0105)

# Baseline: три обучения epochs fix, сравнение и выбор optimizator Adam, RMSProp и SGD

In [ ]:
optimizers = {
    "adam": Adam(learning_rate=1e-3),
    "rmsprop": RMSprop(learning_rate=1e-3),
    "sgd": SGD(learning_rate=1e-2, momentum=0.0)
}

BEST_EPOCHS = 10

for opt_name, opt_instance in optimizers.items():
    # создаем и компилируем модель заново (чтобы не делать повторное обучение на ранее полученных весах)
    model = build_model(input_shape)
    model.compile(
        loss="binary_crossentropy",
        optimizer=opt_instance,
        metrics=["accuracy"]
    )
    
    # обучение на train_generator с валидацией на val_generator
    model.fit(
        train_generator,
        epochs=BEST_EPOCHS,
        validation_data=val_generator,
        verbose=1
    )
    
    # оценка на тестовом генераторе
    loss, accuracy = model.evaluate(test_generator, verbose=0)
    print(f"Точность на тесте с {opt_name}: {accuracy:.4f} / Потери: {loss:.4f}")

c:\Users\JohnOnGear\Desktop\Nut_Classify_CNN\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 194s 482ms/step - accuracy: 0.5148 - loss: 0.6956 - val_accuracy: 0.5000 - val_loss: 0.6450
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 192s 479ms/step - accuracy: 0.8177 - loss: 0.4028 - val_accuracy: 0.7800 - val_loss: 0.6523
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 189s 471ms/step - accuracy: 0.8963 - loss: 0.2812 - val_accuracy: 0.9300 - val_loss: 0.1989
Epoch 4/10
149/400 ━━━━━━━━━━━━━━━━━━━━ 1:43 414ms/step - accuracy: 0.9396 - loss: 0.1640

# Final model (epoch = 10, optimizator = )

In [ ]:
BEST_OPTIMIZER = tf.keras.optimizers.Adam(learning_rate=1e-3)
BEST_EPOCHS = 10

# создаем и компилируем финальную модель
final_model = build_model(input_shape)
final_model.compile(
    loss="binary_crossentropy",
    optimizer=BEST_OPTIMIZER,
    metrics=["accuracy"]
)

# объединяем train + val в единый генератор, чтобы использовать больший массив данных для финального обучения
train_val_generator = SimpleImageDataset(
    np.concatenate([X_train, X_val]),
    np.concatenate([y_train, y_val]),
    batch_size=batch_size,
    shuffle=True
)

final_model.fit(
    train_val_generator,
    epochs=BEST_EPOCHS,
    verbose=1
)

# оценка на тесте
loss, accuracy = final_model.evaluate(test_generator, verbose=0)
print(f"Финальный тест accuracy: {accuracy:.4f} / loss: {loss:.4f}")

# сохранение модели
os.makedirs('artefacts', exist_ok=True)
final_model.save("artefacts/final_model.keras")

# Интерпретация результатов